In [15]:
import numpy as np 
import pandas as pd
import random

In [16]:
def generate_data():

    INTENTS = ["attack", "retreat", "reposition", "idle"]
    ACTIONS = ["center_flank", "from_left_flank", "from_right_flank", "moving_back"]

    TYPE_WEIGHTS = {"TANK": 1.0, "IFV": 0.8, "APC": 0.7}
    ACTION_WEIGHTS = {
        "center_flank": 1.0,
        "from_left_flank": 0.9,
        "from_right_flank": 0.9,
        "moving_back": 0.3
    }
    INTENT_WEIGHTS = {
        "attack": 1.0,
        "reposition": 0.7,
        "retreat": 0.3,
        "idle": 0.1
    }

    MAX_DIST = 1000

    CLASSES = ["TANK", "IFV", "APC"]


    # ---------------------------------------------
    cls = random.choice(CLASSES)
    type_score = TYPE_WEIGHTS[cls]
    # ---------------------------------------------

    X = random.uniform(-200, 200)
    Y = random.uniform(5, 500)

    vx = random.uniform(-5, 5)
    vy = random.uniform(-5, 5)

    X += np.random.normal(0, 5)
    Y += np.random.normal(0, 5)

    distance = np.sqrt(X**2 + Y**2)
    distance_score = max(0, 1 - distance / MAX_DIST)
    distance_score += np.random.normal(0, 0.02)
    distance_score = np.clip(distance_score, 0, 1)    

    # ---------------------------------------------

    future_Y = Y + vy * 2
    future_dist_score = max(0, 1 - future_Y / MAX_DIST)
    
    future_dist_score += np.random.normal(0, 0.02)
    future_dist_score = np.clip(future_dist_score, 0, 1)

    # ---------------------------------------------

    action = random.choice(ACTIONS)
    action_score = ACTION_WEIGHTS[action]

    action_proba = random.uniform(0.3, 1.0)
    action_proba += np.random.normal(0, 0.05)
    action_proba = np.clip(action_proba, 0, 1)

    action_score *= action_proba

    # ---------------------------------------------
   

    if abs(vx) + abs(vy) < 0.2:
        direction_score = 0.5
    elif vy < 0:
        direction_score = 1.0
    elif abs(vx) > abs(vy):
        direction_score = 0.7
    else:
        direction_score = 0.3

    # ---------------------------------------------

    conf = random.uniform(0.5, 1.0)
    conf += np.random.normal(0, 0.05)
    conf = np.clip(conf, 0, 1)

    # ---------------------------------------------

    intent = random.choice(INTENTS)
    intent_score = INTENT_WEIGHTS[intent]

    threat = (
        0.2 * type_score +
        0.15 * distance_score +
        0.15 * future_dist_score +
        0.15 * action_score +
        0.1 * direction_score +
        0.1 * conf +
        0.2 * intent_score
    )

    threat += np.random.normal(0, 0.03)
    threat = np.clip(threat, 0, 1)

    return [type_score, distance_score, future_dist_score, action_score, direction_score, conf, intent_score, threat]


In [17]:
data = [generate_data() for _ in range(10_000)]

In [18]:
columns = [
    "class_score",              
    "distance_score",
    "future_dist_score",
    "action",
    "direction_score",
    "confidence",
    "intent",
    "threat"
]

data = pd.DataFrame(data, columns=columns)

In [19]:
data

,class_score,distance_score,future_dist_score,action,direction_score,confidence,intent,threat
0,0.8,0.635625,0.655069,0.900000,0.3,0.912528,1.0,0.826497
1,0.8,0.648480,0.683309,0.789551,1.0,1.000000,0.1,0.709557
2,0.8,0.834844,0.974994,0.474349,1.0,0.800161,0.7,0.834134
3,1.0,0.739883,0.767081,0.659475,1.0,0.807713,0.7,0.839423
4,0.8,0.831930,0.907672,0.642666,1.0,0.921371,0.3,0.775294
...,...,...,...,...,...,...,...,...
9995,0.7,0.886840,0.910355,0.841602,1.0,0.784648,0.1,0.752619
9996,0.7,0.561338,0.564383,0.775157,0.3,0.735958,1.0,0.766956
9997,0.8,0.874913,0.969492,0.350685,0.7,1.000000,1.0,0.780749
9998,0.8,0.593956,0.583294,0.437061,1.0,0.967550,0.1,0.611748


In [20]:
min(data['distance_score'])

0.42942935790337544

In [21]:
data.to_csv('threat_data.csv')